# Student Performance Prediction

Predict a student's **Performance Index** from study habits with a reproducible
scikit-learn regression workflow.

**Workflow:** data validation → exploratory analysis → preprocessing → model
comparison → cross-validation → interpretation → final model export.

> Expected input: `data/student_performance_dirty.csv`. The dataset is not
> included in this repository; place your local copy at that path before running.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
DATA_PATH = Path("data/student_performance_dirty.csv")
MODEL_PATH = Path("models/student_performance_model.joblib")
TARGET = "Performance Index"

## 1. Load the data 📥

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 2. First look 🔍

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 3. Cleaning 🧹

- **Duplicates are removed *before* splitting.** Otherwise the same row could land in both train and test and inflate the score (data leakage).
- Numeric columns are coerced to numbers, and values outside a realistic range become `NaN` (they get imputed later).
- `Sample Question Papers Practiced` takes the values 0–9 with roughly 1,000 rows each, but a handful of rows contain `27`. That is clearly a data-entry error, so it is treated as missing instead of dropping the whole row.

In [ ]:
VALID_RANGES = {
    "Hours Studied": (0, 24),
    "Previous Scores": (0, 100),
    "Sleep Hours": (0, 24),
    "Sample Question Papers Practiced": (0, 9),
}

df = df.drop_duplicates().reset_index(drop=True)

for col, (low, high) in VALID_RANGES.items():
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[~df[col].between(low, high), col] = np.nan

print(df.shape)
print(df.isnull().sum())

## 4. Exploration 📊

In [ ]:
print(df["Extracurricular Activities"].value_counts(dropna=False))
print()
print(df.describe())

**Is the `Notes` column useful?** If it carried information, the average score would differ clearly between its two values.

In [ ]:
print(df.groupby("Notes")[TARGET].agg(["mean", "std", "count"]))

The two groups differ by well under one point while the standard deviation is about 19, so `Notes` carries no signal. It is dropped to avoid feeding the model noise.

In [ ]:
df = df.drop(columns=["Notes"])

corr = (
    df.assign(Extra=(df["Extracurricular Activities"] == "Yes").astype(int))
    .drop(columns=["Extracurricular Activities"])
    .corr()
)
corr[TARGET].sort_values(ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["Previous Scores"], df[TARGET], alpha=0.2, s=12)
ax.set_xlabel("Previous Scores")
ax.set_ylabel(TARGET)
ax.set_title("Previous Scores vs Performance Index")
ax.grid(alpha=0.2)
plt.show()

## 5. Train / test split ✂️

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 6. Preprocessing pipeline 🛠️

Imputation and encoding live **inside** the pipeline, so they are fitted on training data only (no leakage, also during cross-validation).

In [ ]:
NUM_COLS = ["Hours Studied", "Previous Scores", "Sleep Hours", "Sample Question Papers Practiced"]
CAT_COLS = ["Extracurricular Activities"]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
    ("cat", OneHotEncoder(drop="if_binary"), CAT_COLS),
])

## 7. Model comparison 🏁

A dummy model (always predicts the training mean) is the baseline every real model has to beat.

In [ ]:
models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest (regularised)": RandomForestRegressor(
        n_estimators=200, max_depth=6, min_samples_leaf=20,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=3,
        subsample=0.8, random_state=RANDOM_STATE,
    ),
}

rows = []
fitted = {}
for name, estimator in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    for split, X_s, y_s in [("train", X_train, y_train), ("test", X_test, y_test)]:
        pred = pipe.predict(X_s)
        rows.append({
            "model": name,
            "split": split,
            "MAE": mean_absolute_error(y_s, pred),
            "RMSE": np.sqrt(mean_squared_error(y_s, pred)),
            "R2": r2_score(y_s, pred),
        })

results = pd.DataFrame(rows).pivot(index="model", columns="split").round(4)
results

**Takeaway:** the relationship is almost linear, so the simple linear model matches or beats the tree ensembles. A more complex model is not automatically a better one.

## 8. Cross-validation ✅

In [ ]:
best = Pipeline([("prep", preprocess), ("model", LinearRegression())])
scores = cross_val_score(best, X, y, cv=5, scoring="r2")
print(scores.round(4))
print(f"Mean R2: {scores.mean():.4f}  (std {scores.std():.4f})")

## 9. What did the model learn? 🧠

Each coefficient is the change in score for **one extra unit** of that feature, holding the others fixed.

In [ ]:
linreg = fitted["Linear Regression"]
coef = pd.Series(
    linreg.named_steps["model"].coef_,
    index=linreg.named_steps["prep"].get_feature_names_out(),
).sort_values(key=abs, ascending=False)
coef.round(3)

## 10. Actual vs predicted 🎯

In [ ]:
pred = linreg.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, pred, alpha=0.3, s=12)
ax.plot([10, 100], [10, 100], "r--", label="perfect prediction")
ax.set_xlabel("Actual Performance Index")
ax.set_ylabel("Predicted Performance Index")
ax.set_title("Linear Regression on the test set")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 11. Save the final model 💾

The final model is refit on **all** the data (model selection is already done), then saved with `joblib`.

In [ ]:
final_model = Pipeline([("prep", preprocess), ("model", LinearRegression())])
final_model.fit(X, y)

MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(final_model, MODEL_PATH)

new_student = pd.DataFrame([{
    "Hours Studied": 6,
    "Previous Scores": 80,
    "Extracurricular Activities": "Yes",
    "Sleep Hours": 7,
    "Sample Question Papers Practiced": 4,
}])
print(f"Predicted Performance Index: {final_model.predict(new_student)[0]:.1f}")

## 12. Limitations and next steps

- The evaluation is valid only for data that follows the same collection process.
- `Notes` is excluded based on the observed exploratory comparison; revisit this
  decision if the column's meaning or data-generation process changes.
- Consider residual analysis, permutation importance, and a held-out external
  dataset before using predictions in a real educational setting.
- Predictions describe patterns in this dataset; they should not be used as the
  sole basis for decisions about individual students.
